In [ ]:
!pip install datasets pandas --quiet

In [15]:
from datasets import load_dataset
import pandas as pd

dataset = load_dataset("vg055/RestMex2023_review-corpus_DataAugV1")
print(dataset)

df = dataset["train"].to_pandas()
print("\nShape:", df.shape)
print("\nColumnas:", df.columns.tolist())
print("\nPrimeras filas:")
df.head(3)

README.md:   0%|          | 0.00/384 [00:00<?, ?B/s]

data/train-00000-of-00001-769ebbca058354(…):   0%|          | 0.00/74.2M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/332823 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 332823
    })
})

Shape: (332823, 1)

Columnas: ['text']

Primeras filas:


,text
0,Es un museo ENORME. Si quieres verlo con deten...
1,Es un lugar que por su arquitectura e historia...
2,Este museo reúne piezas invaluables y reconstr...


In [16]:
dataset2 = load_dataset("javilonso/restmex23-test")
df2 = dataset2["train"].to_pandas()

print("Shape:", df2.shape)
print("Columnas:", df2.columns.tolist())
print(df2.head(3))

README.md:   0%|          | 0.00/399 [00:00<?, ?B/s]

data/train-00000-of-00001-f230596628d495(…):   0%|          | 0.00/27.7M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/107863 [00:00<?, ? examples/s]

Shape: (107863, 2)
Columnas: ['ID', 'Title_Review']
   ID                                       Title_Review
0   0  Excelente servicio, buenas instalaciones, limp...
1   1  Lo pasamos genial Pasamos unos días muy agrada...
2   2  Muy congestionado Demasiada gente en un vagón ...


In [17]:
dataset3 = load_dataset("vg055/RestMex2023_review-corpus_DataAugV2")
df3 = dataset3["train"].to_pandas()

print("Shape:", df3.shape)
print("Columnas:", df3.columns.tolist())
print(df3.head(3))

README.md:   0%|          | 0.00/480 [00:00<?, ?B/s]

data/train-00000-of-00001-fdaa48157ac222(…):   0%|          | 0.00/66.0M [00:00<?, ?B/s]

data/test-00000-of-00001-013c3dbaf26a36c(…):   0%|          | 0.00/6.41M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/265723 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25171 [00:00<?, ? examples/s]

Shape: (265723, 2)
Columnas: ['text', 'label']
                                                text  label
0  La mejor relación calidad-precio Deliciosa com...      5
1  Nos sentimos especiales El 28 de enero era nue...      5
2  AUSTERO, ANTIGUO, BIEN UBICADO, A UNA CUADRA D...      3


In [18]:
from datasets import load_dataset
import pandas as pd

dataset = load_dataset("vg055/RestMex2023_review-corpus_DataAugV2")
df = dataset["train"].to_pandas()

# Distribución de etiquetas
print("Distribución de polaridad:")
print(df["label"].value_counts().sort_index())

# Longitud de textos
df["text_len"] = df["text"].str.len()
print("\nLongitud de texto:")
print(df["text_len"].describe())

# Muestra de cada clase
print("\nEjemplo por clase:")
for label in sorted(df["label"].unique()):
    ejemplo = df[df["label"] == label].iloc[0]["text"][:100]
    print(f"\n[{label}] {ejemplo}...")

Distribución de polaridad:
label
1     18002
2     16631
3     35501
4     54204
5    141385
Name: count, dtype: int64

Longitud de texto:
count    265723.000000
mean        396.211209
std         415.326712
min           9.000000
25%         202.000000
50%         280.000000
75%         458.000000
max       19921.000000
Name: text_len, dtype: float64

Ejemplo por clase:

[1] Muy mal producto En relación a los platos, tuvieron muy mala cocción. El rack de cordero pésimo, tan...

[2] No vale el dinero pagado Viajo mucho por trabajo y hacía tiempo que no me sentía tan incomoda en un ...

[3] AUSTERO, ANTIGUO, BIEN UBICADO, A UNA CUADRA DEL CAPITOLIO, BUENA ATENCION Es un hotel pintoresco, b...

[4] El favorito en La Habana Nuestra cena en Dona Eutimia fue una de nuestras cenas favoritas en La Haba...

[5] La mejor relación calidad-precio Deliciosa comida cubana, en excelente ambiente y muy buen servicio....


In [19]:
df.to_csv("../data/raw/restmex_train.csv", index=False)
print(f"Guardado: {df.shape[0]} filas en data/raw/restmex_train.csv")

Guardado: 265723 filas en data/raw/restmex_train.csv


In [20]:
# Convertir 1-5 a positivo/negativo/neutro
def map_label(x):
    if x in [1, 2]:
        return "negativo"
    elif x == 3:
        return "neutro"
    else:  # 4, 5
        return "positivo"

df["sentiment"] = df["label"].map(map_label)

print("Distribución tras mapeo:")
print(df["sentiment"].value_counts())

# Renombrar columnas al esquema del normalizer
df = df.rename(columns={"text": "texto"})
df["rating"]     = df["label"]        # guardamos el original
df["plataforma"] = "rest-mex"
df["fecha"]      = None

print("\nColumnas finales:", df.columns.tolist())
print(df.head(3))

Distribución tras mapeo:
sentiment
positivo    195589
neutro       35501
negativo     34633
Name: count, dtype: int64

Columnas finales: ['texto', 'label', 'text_len', 'sentiment', 'rating', 'plataforma', 'fecha']
                                               texto  label  text_len  \
0  La mejor relación calidad-precio Deliciosa com...      5       149   
1  Nos sentimos especiales El 28 de enero era nue...      5       813   
2  AUSTERO, ANTIGUO, BIEN UBICADO, A UNA CUADRA D...      3       594   

  sentiment  rating plataforma fecha  
0  positivo       5   rest-mex  None  
1  positivo       5   rest-mex  None  
2    neutro       3   rest-mex  None  


In [21]:
df.to_csv("../data/raw/restmex_train.csv", index=False)
print(f"Guardado: {df.shape[0]} filas")

Guardado: 265723 filas


In [22]:
df = df.drop(columns=["text_len"])

df.to_csv("../data/raw/restmex_train.csv", index=False)
print(f"✓ Guardado: {df.shape[0]} filas, {df.shape[1]} columnas")
print(df.columns.tolist())

✓ Guardado: 265723 filas, 6 columnas
['texto', 'label', 'sentiment', 'rating', 'plataforma', 'fecha']


In [24]:
n = 60_000

df_neg = df[df["sentiment"] == "negativo"].sample(n, replace=True, random_state=42)
df_neu = df[df["sentiment"] == "neutro"].sample(n, replace=True, random_state=42)
df_pos = df[df["sentiment"] == "positivo"].sample(n, replace=False, random_state=42)

df_balanced = pd.concat([df_neg, df_neu, df_pos]).sample(frac=1, random_state=42)

print("Distribución balanceada:")
print(df_balanced["sentiment"].value_counts())
print(f"\nTotal filas: {df_balanced.shape[0]}")

Distribución balanceada:
sentiment
positivo    60000
negativo    60000
neutro      60000
Name: count, dtype: int64

Total filas: 180000


In [26]:
import os

# Mover a processed
df_balanced.to_csv("../data/processed/restmex_train.csv", index=False)
df.to_csv("../data/raw/restmex_train_original.csv", index=False)

print("✓ Raw:       data/raw/restmex_train_original.csv")
print("✓ Processed: data/processed/restmex_train.csv")

✓ Raw:       data/raw/restmex_train_original.csv
✓ Processed: data/processed/restmex_train.csv
